# Step 03 - Vacancy Controls

Apply vacancy settings and produce final tensile inputs: `init.vasp`, `bottom_idx.npy`, `top_idx.npy`.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "main.py").exists() and (candidate / "al.gga.psp").exists():
            return candidate
    raise RuntimeError("Project root not found (expected main.py and al.gga.psp).")


ROOT = find_project_root(Path.cwd())
CASE_NAME = "nc_large_vac01"
CASE_DIR = ROOT / "cases" / CASE_NAME
INPUTS_DIR = CASE_DIR / "inputs"

RAW_XYZ = INPUTS_DIR / "raw_structure.xyz"
RAW_BOTTOM = INPUTS_DIR / "raw_bottom_idx.npy"
RAW_TOP = INPUTS_DIR / "raw_top_idx.npy"

assert RAW_XYZ.exists(), f"Missing: {RAW_XYZ}"
assert RAW_BOTTOM.exists(), f"Missing: {RAW_BOTTOM}"
assert RAW_TOP.exists(), f"Missing: {RAW_TOP}"

# Vacancy controls
ENABLE_VACANCY = True
VAC_MODE = "conc"       # one | count | conc
VAC_N = 3
VAC_CONC_PCT = 0.1      # used when VAC_MODE=conc
VAC_CONC_BASIS = "free" # total | free
VAC_REGION = "free"     # free | all
SEED = 42


In [ ]:
if ENABLE_VACANCY:
    cmd = [
        sys.executable,
        str(ROOT / "make_vacancy.py"),
        "--input", "raw_structure.xyz",
        "--bottom", "raw_bottom_idx.npy",
        "--top", "raw_top_idx.npy",
        "--tag", "init",
        "--seed", str(SEED),
        "--mode", VAC_MODE,
        "--n", str(VAC_N),
        "--conc-pct", str(VAC_CONC_PCT),
        "--conc-basis", VAC_CONC_BASIS,
        "--region", VAC_REGION,
    ]
    subprocess.run(cmd, check=True, cwd=str(INPUTS_DIR))

    shutil.copy2(INPUTS_DIR / "bottom_idx_init.npy", INPUTS_DIR / "bottom_idx.npy")
    shutil.copy2(INPUTS_DIR / "top_idx_init.npy", INPUTS_DIR / "top_idx.npy")
else:
    shutil.copy2(RAW_XYZ, INPUTS_DIR / "init.xyz")
    shutil.copy2(INPUTS_DIR / "raw_structure.vasp", INPUTS_DIR / "init.vasp")
    shutil.copy2(RAW_BOTTOM, INPUTS_DIR / "bottom_idx.npy")
    shutil.copy2(RAW_TOP, INPUTS_DIR / "top_idx.npy")

print("Init structure:", INPUTS_DIR / "init.vasp")
print("Bottom idx    :", INPUTS_DIR / "bottom_idx.npy")
print("Top idx       :", INPUTS_DIR / "top_idx.npy")


In [ ]:
import numpy as np
from ase.io import read

atoms = read(str(INPUTS_DIR / "init.xyz"))
b = np.load(str(INPUTS_DIR / "bottom_idx.npy"))
t = np.load(str(INPUTS_DIR / "top_idx.npy"))

print("Atoms after Step 03:", len(atoms))
print("Bottom atoms:", len(b), "Top atoms:", len(t))
